In [1]:
import uuid
from datetime import datetime

PIPELINE_NAME = "04_6_gold_city_metrics"
LAYER = "gold"
TABLE_NAME = "city_metrics_gold"
LOG_TABLE = "pipeline_run_log"

run_id = str(uuid.uuid4())
start_time = datetime.now()

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    pipeline_run_id STRING,
    pipeline_name STRING,
    layer STRING,
    table_name STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    input_rows BIGINT,
    output_rows BIGINT,
    dq_status STRING
)
USING DELTA
""")

def log_start():
    spark.sql(f"""
    INSERT INTO  {LOG_TABLE}
    VALUES (
        '{run_id}',
        '{PIPELINE_NAME}',
        '{LAYER}',
        '{TABLE_NAME}',
        'RUNNING',
        CURRENT_TIMESTAMP(),
        NULL,
        NULL,
        NULL,
        NULL

    )
    """)

def log_finish(status, input_rows=None, output_rows=None,dq_status=None):

    input_rows_sql = "NULL" if input_rows is None else str(input_rows)
    output_rows_sql = "NULL" if output_rows is None else str(output_rows)
    dq_status_sql = "NULL" if dq_status is None else str(dq_status)

    spark.sql(f"""
    MERGE INTO {LOG_TABLE}  t
    USING (
        SELECT '{PIPELINE_NAME}' AS pipeline_name,
        '{run_id}' AS pipeline_run_id
    ) s
    ON t.pipeline_name = s.pipeline_name
    AND t.pipeline_run_id = s.pipeline_run_id
    WHEN MATCHED THEN UPDATE SET
        t.status = '{status}',
        t.end_time = CURRENT_TIMESTAMP(),
        t.input_rows = {input_rows_sql},
        t.output_rows = {output_rows_sql},
        t.dq_status = '{dq_status_sql}'
    """)

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


# ================= build gold =================

BUSINESS_METRICS = "business_metrics_gold"
GOLD = "city_metrics_gold"

df = spark.table(BUSINESS_METRICS)

df_city = (
    df.groupBy("state", "city")
    .agg(
        F.count("business_id").alias("business_count"),
        F.sum("review_count_total").alias("review_count_total"),
        F.round(
            F.sum(F.col("avg_rating") * F.col("review_count_total")) / F.sum(F.col("review_count_total")),
            2
        ).alias("weighted_avg_rating")
    )
)




StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 4, Finished, Available, Finished, False)

In [3]:
input_rows = df.count()

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 5, Finished, Available, Finished, False)

In [4]:
# ======================== DQ start ========================== #

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 6, Finished, Available, Finished, False)

In [5]:
output_rows = df_city.count()

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 7, Finished, Available, Finished, False)

In [6]:
def run_gold_city_metrics_dq(df):

    total_rows = df.count()
    results = []

    def add_result(
        rule_name, 
        dimension, 
        column_name, 
        failed_count, 
        threshold, 
        notes=""
        ):
        status = "PASS" if failed_count <= threshold else "FAIL"
        results.append({
            "rule_name": rule_name, 
            "dimension": dimension, 
            "column_name": column_name, 
            "failed_count": failed_count, 
            "threshold": threshold, 
            "status": status,
            "notes": notes
        })


    #1. row count
    add_result(
        rule_name = "row_count_gt_0",
        dimension = "completeness",
        column_name = "*",
        failed_count = 0 if total_rows > 0 else 1,
        threshold = 0,
        notes = f"total_rows={total_rows}"
    )

    #2.city not null
    null_city = df.filter(F.col("city").isNull()).count()
    add_result(
        rule_name = "city_not_null", 
        dimension = "completeness", 
        column_name = "city", 
        failed_count = null_city, 
        threshold =  0
    )

    #3.state not null
    null_state = df.filter(F.col("state").isNull()).count()
    add_result(
        rule_name = "state_not_null", 
        dimension = "completeness", 
        column_name = "state", 
        failed_count = null_state, 
        threshold =  0
    )
    #4. city-state unique
    dup_city_state = (
        df.groupBy("city", "state")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    add_result(
        rule_name = "city_state_unique", 
        dimension = "uniqueness", 
        column_name = "city, state", 
        failed_count = dup_city_state, 
        threshold = 0 
    )

    #5. business_count >= 0
    bad_business_count = df.filter(F.col("business_count") < 0).count()
    add_result(
        rule_name = "business_count_gte_0", 
        dimension = "validity", 
        column_name = "business_count", 
        failed_count = bad_business_count, 
        threshold = 0
    )

    #6. review_total >=0
    bad_reviews = df.filter(F.col("review_count_total") < 0 ).count()
    add_result(
        rule_name = "reviews_total_gte_0", 
        dimension = "validity", 
        column_name = "review_count_total", 
        failed_count = bad_reviews, 
        threshold = 0
    )

    #7.avg_rating between 1 and 5
    bad_rating = df.filter((F.col("weighted_avg_rating") < 1) | (F.col("weighted_avg_rating") > 5)).count() 
    add_result(
        rule_name = "avg_rating_between_1_and_5", 
        dimension = "validity", 
        column_name = "weighted_avg_rating", 
        failed_count = bad_rating, 
        threshold = 0
    )

    dq_df = spark.createDataFrame(results)

    fail_count = dq_df.filter(F.col("status") == "FAIL").count()
    display(dq_df.orderBy("status","rule_name"))

    if fail_count > 0:
        raise Exception("Gold DQ failed for city_metrics_gold")
    print("Gold DQ passed for city_metrics_gold")

    return dq_df

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 8, Finished, Available, Finished, False)

In [7]:
dq_result = run_gold_city_metrics_dq(df_city)

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d5525d7-8979-431a-b235-3b8a39338d04)

Gold DQ passed for city_metrics_gold


In [8]:
dq_status = "PASSED"

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 10, Finished, Available, Finished, False)

In [9]:
# ======================== DQ end ========================== #

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 11, Finished, Available, Finished, False)

In [12]:
# ================= write table =================
try:

    log_start()

    if spark.catalog.tableExists(GOLD):
        delta = DeltaTable.forName(spark, GOLD)

        (
            delta.alias("t")
            .merge(
                df_city.alias("c"),
                "t.state = c.state and t.city = c.city"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            df_city.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(GOLD)
        )
    
    log_finish(
    status = "SUCCESS",
    input_rows = input_rows,
    output_rows = output_rows,
    dq_status = dq_status
    )
except Exception as e:
    log_finish(
        status = "FAILED",
        input_rows = input_rows,
        output_rows = output_rows,
        dq_status = "FAILED"
    )
    raise

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 14, Finished, Available, Finished, False)

In [13]:
display(spark.table("pipeline_run_log"))

StatementMeta(, 14a7ac9b-b254-482b-9265-7db98b7ee29a, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ffd36891-02c5-47b2-8bb6-ae2866030dcb)